# Word-Source Dictionary (Trie)
- update: 2023-10-03

In [1]:
import re

class Word_Source_Dictionary(): # update: 2023-10-03
    def __init__(self, word_tokenize=lambda s: re.findall('[0-9a-zA-Z]+', s), is_case_sensitive=False, search_trailing_substring=True):
        self.word_tokenize = word_tokenize
        self.is_case_sensitive = is_case_sensitive
        self.search_trailing_substring = search_trailing_substring
        
        self.trie = {} # initialize trie data structure
        
        self.EOW = '##' # the end of word
        assert len(self.EOW) > 1, 'MUST be more than one character...' # see REF #1
        self.SRC = '$$' # a set of sources
        assert len(self.SRC) > 1, 'MUST be more than one character...' # see REF #1
        self._SRC = '_$' # a set of sources, for search_trailing_substring
        assert len(self._SRC) > 1, 'MUST be more than one character...' # see REF #1

    def _add_word(self, word, source, TEG_SRC):
        node = self.trie
        for c in word:
            if c not in node:
                node[c] = {} # create if none, to proceed
            node = node[c]

        node[self.EOW] = word # the end of word. overwrite even if there exists any
        node.setdefault(TEG_SRC, set()).add(source) # a set of sources
        return
        
    def add(self, text, source):
        assert isinstance(text, str) and len(text) > 0, 'Brad error: only non-empty string is allowed....'

        words = self.word_tokenize(text if self.is_case_sensitive else text.lower())
        for word in words:
            for i in range(len(word) if self.search_trailing_substring else 1): # ex) apple, _pple, __ple, ___le, ____e
                _word = word[i:]
                
                if i == 0:
                    self._add_word(_word, source, self.SRC)
                else:
                    self._add_word(_word, source, self._SRC)
        return

    def _find_all_leaves(self, node):
        leaves = []
        for k in node.keys():
            if len(k) == 1: # only if a single character, REF #1
                leaves.append(node[k])
                leaves.extend(self._find_all_leaves(node[k])) # recursive search
        return leaves

    def _find_all_common_roots(self, word):
        word = word if self.is_case_sensitive else word.lower()
        
        roots = []
        node = self.trie
        for c in word:
            if c not in node:
                return roots
            node = node[c]
            roots.append(node)

        return roots

    def _find_sources_for_word(self, word, allow_suffix=True, allow_prefix=True, allow_inner_suffix=False, allow_inner_prefix=False):
        word = word if self.is_case_sensitive else word.lower()
        
        sources = set()

        #--- target node and leaf nodes -------------------
        roots = self._find_all_common_roots(word)
        
        if len(roots) == len(word): # ensures the target match
            leaves = self._find_all_leaves(roots[-1]) if allow_suffix else [] # allow matching like xxx_, xxx__, xxx___, ....

            for node in roots[-1:] + leaves: # target node + leaf nodes (if any)
                if self.SRC in node:
                    sources = sources.union(node[self.SRC]) # only extract matching
                    
                if allow_prefix and (self._SRC in node): # allow matching like _xxx, __xxx, ___xxx, ....
                    sources = sources.union(node[self._SRC])

        #--- find root nodes ---------------------
        if allow_inner_suffix or allow_inner_prefix:
            max_len = len(word)
            
            for i in range(max_len if allow_inner_prefix else 1): # _yyy, __yy, ___y
                _word = word[i:]
                
                roots = self._find_all_common_roots(_word)
                roots_ = roots[:(max_len-1)] # exclude only the target node (i.e. word), if any (cf. if len(roots) < len(word), then no effect)

                if allow_inner_suffix:
                    for node in roots_: 
                        if self.SRC in node:
                            sources = sources.union(node[self.SRC])
                elif (len(roots_) == len(_word)) and (self.SRC in node):
                    sources = sources.union(node[self.SRC])

        return sources

    def find_sources(self, texts, allow_suffix=True, allow_prefix=True, allow_inner_suffix=False, allow_inner_prefix=False):
        assert isinstance(texts, list) and len(texts) > 0, 'Brad error: only a non-empty list of texts is allowed....'

        sources = set()
        for text in texts:
            words = self.word_tokenize(text if self.is_case_sensitive else text.lower())

            if len(words) == 0:
                ss = set()
                break

            ss = self._find_sources_for_word(words[0], allow_suffix=allow_suffix, allow_prefix=allow_prefix, allow_inner_suffix=allow_inner_suffix, allow_inner_prefix=allow_inner_prefix)
            for word in words[1:]:
                ss = ss.intersection(self._find_sources_for_word(word, allow_suffix=allow_suffix, allow_prefix=allow_prefix, allow_inner_suffix=allow_inner_suffix, allow_inner_prefix=allow_inner_prefix))

            sources = sources.union(ss)

        return sources


if __name__ == "__main__":
    wsd = Word_Source_Dictionary()

    for text, source in [('Apple Banana',1), ('Cherry',1), ('Cat Dog',2), ('door',3), ('app',4), ('PPL',5), ('TheApplePie',6)]:
        wsd.add(text, source)

    print(wsd.trie)
    # wsd.trie['p']['p']

    print(wsd.find_sources(['apple banana'])) # 'apple' AND 'banana'
    print(wsd.find_sources(['apple', 'door'])) # 'apple' OR 'door'
    print(wsd.find_sources(['apple door'])) # 'apple' AND 'door'

    print(wsd.find_sources(['pp']))
    print(wsd.find_sources(['app'], allow_prefix=False, allow_suffix=False)) # exact matching
    print(wsd.find_sources(['apple']))
    print(wsd.find_sources(['apple'], allow_prefix=False, allow_suffix=False, allow_inner_suffix=True, allow_inner_prefix=True))
    print(wsd.find_sources(['apple'], allow_prefix=True, allow_suffix=True, allow_inner_suffix=True, allow_inner_prefix=True))


{'a': {'p': {'p': {'l': {'e': {'##': 'apple', '$$': {1}, 'p': {'i': {'e': {'##': 'applepie', '_$': {6}}}}}}, '##': 'app', '$$': {4}}}, 'n': {'a': {'n': {'a': {'##': 'anana', '_$': {1}}}, '##': 'ana', '_$': {1}}}, '##': 'a', '_$': {1}, 't': {'##': 'at', '_$': {2}}}, 'p': {'p': {'l': {'e': {'##': 'pple', '_$': {1}, 'p': {'i': {'e': {'##': 'pplepie', '_$': {6}}}}}, '##': 'ppl', '$$': {5}}, '##': 'pp', '_$': {4}}, 'l': {'e': {'##': 'ple', '_$': {1}, 'p': {'i': {'e': {'##': 'plepie', '_$': {6}}}}}, '##': 'pl', '_$': {5}}, '##': 'p', '_$': {4}, 'i': {'e': {'##': 'pie', '_$': {6}}}}, 'l': {'e': {'##': 'le', '_$': {1}, 'p': {'i': {'e': {'##': 'lepie', '_$': {6}}}}}, '##': 'l', '_$': {5}}, 'e': {'##': 'e', '_$': {1, 6}, 'r': {'r': {'y': {'##': 'erry', '_$': {1}}}}, 'a': {'p': {'p': {'l': {'e': {'p': {'i': {'e': {'##': 'eapplepie', '_$': {6}}}}}}}}}, 'p': {'i': {'e': {'##': 'epie', '_$': {6}}}}}, 'b': {'a': {'n': {'a': {'n': {'a': {'##': 'banana', '$$': {1}}}}}}}, 'n': {'a': {'n': {'a': {'##': '